# Introduction

The purpose of this notebook is to investigate how we can create a nearest neighbour similarity based approach for comparing listings. 

In [9]:
from pandas import read_sql

from postgre_tools import Users, create_postgres_engine


In [10]:
engine = create_postgres_engine(Users.AIRBNB.load_user())

# Set count to None if you want to fetch all rows
listings_row_count: int | None = None
amenities_row_count: int | None = None

if listings_row_count is not None and listings_row_count <= 0:
    raise ValueError("listings_row_count must be greater than 0")
if amenities_row_count is not None and amenities_row_count <= 0:
    raise ValueError("amenities_row_count must be greater than 0")

listings_query = "SELECT id, room_type, accommodates, bedrooms, beds, district FROM listings"
amenities_query = "SELECT * FROM listing_amenities"

if listings_row_count is not None:
    listings_query += f" LIMIT {listings_row_count}"

if amenities_row_count is not None:
    amenities_query += f" LIMIT {amenities_row_count}"

listings_df = read_sql(listings_query, engine)
listing_amenities_df = read_sql(amenities_query, engine)

engine.dispose()

In [16]:
print(f"Columns in listings_df: {listings_df.columns}")
print(f"\nData types in listings_df:\n{listings_df.dtypes}")
listings_df

Columns in listings_df: Index(['id', 'room_type', 'accommodates', 'bedrooms', 'beds', 'district'], dtype='str')

Data types in listings_df:
id                int64
room_type           str
accommodates      int64
bedrooms        float64
beds            float64
district            str
dtype: object


,id,room_type,accommodates,bedrooms,beds,district
0,145320,Private room,2,1.0,1.0,Södermalm
1,164448,Private room,2,NaN,2.0,Södermalm
2,220851,Private room,1,NaN,1.0,Kungsholmen
3,242188,Private room,1,NaN,1.0,Södermalm
4,273906,Entire home/apt,4,3.0,NaN,Södermalm
...,...,...,...,...,...,...
4801,1717861167246979973,Entire home/apt,2,1.0,1.0,Skarpnäck
4802,1718132464242032867,Private room,2,1.0,1.0,Järva
4803,1718263471917793029,Entire home/apt,2,1.0,1.0,Bromma
4804,1718288866708700915,Entire home/apt,2,1.0,1.0,Kungsholmen


The columns, except id, are the ones used as a basis for similarity. room_type and district are categorical, so similarities can be applied by using ==.

In [17]:
listings_df.to_csv("listings_df.csv", index=False)

In [18]:
listing_amenities_df.to_csv("listing_amenities_df.csv", index=False)
listing_amenities_df

,listing_id,amenity
0,1000056802929658315,Bed linens
1,1000056802929658315,Body soap
2,1000056802929658315,Cleaning products
3,1000056802929658315,Coffee
4,1000056802929658315,Coffee maker
...,...,...
103358,999790679792070930,Outdoor dining area
103359,999790679792070930,Refrigerator
103360,999790679792070930,Smoke alarm
103361,999790679792070930,TV
